This notebook performs hyperparameter tuning for all models across all datasets. 

It generates one file per dataset:
``results/runs/<dataset>/opt_params.json``

Each file contains the optimal parameters found for the following models:
- EASE (item- and user-based)
- SLIM (item- and user-based)
- SVD
- NMF
- MF

**Structure of JSON files**:
```json
{
  "model_name": {
    "param_1": value,
    "param_2": value,
    ...
  },
  ...
}

In [ ]:
import os
os.chdir('..')

import json
import numpy as np
from hyperopt import fmin, tpe, hp

# Data and metrics
from src.helper_functions.data_loader import *
from src.helper_functions.data_splitter import *
from src.helper_functions.metrics_accuracy import *

# Models
from src.models.ease import EASE
from src.models.slim import SLIM
from src.models.mf_fair import FairMF
from recpack.algorithms.factorization import SVD, NMF

In [ ]:
DATASETS = ["ml-100k","ml-1m","lastfm-1k","ftky","fnyc"]

LIST_SIZE = 10 # number of recommendations for tuning
RATING_THRES = 4 # rating threshold for defining relevance

SECONDS = 12*3600 # 12 hours
EVALUATIONS = 50
SEED = 42

FACTOR_CHOICES = [8,16,32,64,128,256] # number of latent factors

In [ ]:
def accuracy_objective(model, R_train, R_val, r_thres, n, fit_args={}):
    model.fit(R_train, **fit_args)
    R_hat = model.predict(R_train).toarray()
    R_hat[R_train.nonzero()] = -np.inf # remove observed items
    ndcg, _ = tndcg_at_n(R_hat, R_val, r_thres, n)
    return -ndcg

# Process each dataset
all_opt_params = {}
for dataset in DATASETS:
    all_opt_params[dataset] = {}
    print(f"Processing dataset: {dataset}")

    data, _, _ = load_dataset_by_name(dataset)

    # split the data per user chronologically
    R_train, R_val, _, uid_to_index, iid_to_index = chronological_split_per_user(data)

    # Neighborhood-based models

    # optimize ease-i
    optimisation_results_ease_i = fmin(
        fn=lambda param: accuracy_objective(EASE(l2=param["l2"]), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"l2": hp.loguniform("l2", np.log(1e0), np.log(1e4))},
        algo=tpe.suggest,
        timeout = SECONDS,
        max_evals = EVALUATIONS,
    )

    # optimize ease-u
    optimisation_results_ease_u = fmin(
        fn=lambda param: accuracy_objective(EASE(l2=param["l2"], method="user"), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"l2": hp.loguniform("l2", np.log(1e0), np.log(1e4))},
        algo=tpe.suggest,
        timeout = SECONDS,
        max_evals = EVALUATIONS,
    )

    # optimize slim-i
    optimisation_results_slim_i = fmin(
        fn=lambda param: accuracy_objective(SLIM(l1=param["l1"], l2=param["l2"]), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"l1": hp.loguniform("l1", np.log(1e-3), np.log(50)),
            "l2": hp.loguniform("l2", np.log(1e0), np.log(1e4))},
        algo=tpe.suggest,
        timeout = SECONDS,
        max_evals = EVALUATIONS,
    )

    # optimize slim-u
    optimisation_results_slim_u = fmin(
        fn=lambda param: accuracy_objective(SLIM(l1=param["l1"], l2=param["l2"], method="user"), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"l1": hp.loguniform("l1", np.log(1e-3), np.log(50)),
            "l2": hp.loguniform("l2", np.log(1e0), np.log(1e4))},
        algo=tpe.suggest,
        timeout = SECONDS,
        max_evals = EVALUATIONS,
    )

    # MF-based models

    # optimize svd
    best_svd_params = None
    best_score = float("inf")

    for num_factors in FACTOR_CHOICES:
        score = accuracy_objective(
            SVD(num_components=num_factors, seed=SEED), R_train, R_val, RATING_THRES, LIST_SIZE
        )

        if score < best_score:
            best_score = score
            best_svd_params = {"num_factors": num_factors}

    # optimize nmf
    optimisation_results_nmf = fmin(
        fn=lambda param: accuracy_objective(NMF(num_components=param["num_factors"], alpha=param["alpha"], l1_ratio=param["l1_ratio"], seed=SEED), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"num_factors": hp.choice("num_factors", FACTOR_CHOICES),
            "alpha": hp.loguniform("alpha", np.log(1e-5), np.log(1e2)),
            "l1_ratio": hp.uniform("l1_ratio", 0, 1)},
        algo=tpe.suggest,
        timeout = SECONDS,
        max_evals = EVALUATIONS,
    )
    optimisation_results_nmf["num_factors"] = FACTOR_CHOICES[optimisation_results_nmf["num_factors"]]

    # optimize mf
    optimisation_results_mf = fmin(
        fn=lambda param: accuracy_objective(FairMF(learning_rate=param["learning_rate"], l2=param["l2"], num_factors=param["num_factors"], seed=SEED), R_train, R_val, RATING_THRES, LIST_SIZE),
        space={"learning_rate": hp.loguniform("learning_rate", np.log(1e-6), np.log(1e0)),
            "l2": hp.loguniform("l2", np.log(1e-6), np.log(1e-1)),
            "num_factors": hp.choice("num_factors", FACTOR_CHOICES)
            },
        algo=tpe.suggest,
        timeout=SECONDS,
        max_evals=EVALUATIONS
    )

    optimisation_results_mf["num_factors"] = FACTOR_CHOICES[optimisation_results_mf["num_factors"]]

    # store results
    all_opt_params[dataset] = {
        "ease-i": optimisation_results_ease_i,
        "ease-u": optimisation_results_ease_u,
        "slim-i": optimisation_results_slim_i,
        "slim-u": optimisation_results_slim_u,
        "svd": best_svd_params,
        "mf": optimisation_results_mf,
        "nmf": optimisation_results_nmf,
    }

    folder = f"results/{dataset}"
    os.makedirs(folder, exist_ok=True)
    with open(f"{folder}/opt_params.json", "w") as f:
        json.dump(all_opt_params[dataset], f, indent=4)

print("Optimization completed for all datasets.")